# Week 2 – Day 2: Train/Test Split and Linear Regression Baseline

## Objective

Establish a traditional machine learning baseline for the Real Estate Valuation project using Linear Regression.

The prepared tabular features from Week 2 Day 1 will be divided into training and holdout datasets. A Linear Regression model will then be trained on the training data to establish an initial benchmark for property-price prediction.

This baseline will later be compared with more advanced models such as XGBoost and spatial/graph-based models.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [2]:
# ---------------------------------------------------
# Project Paths
# ---------------------------------------------------

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(PROJECT_ROOT,"data","processed")
MODEL_DIR = os.path.join(PROJECT_ROOT,"models")

os.makedirs(MODEL_DIR, exist_ok=True)

TABULAR_PATH = os.path.join(DATA_DIR,"tabular_features.csv")

print("Project root:")
print(PROJECT_ROOT)

print("\nTabular dataset:")
print(TABULAR_PATH)

Project root:
e:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation

Tabular dataset:
e:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\data\processed\tabular_features.csv


In [3]:
df = pd.read_csv(TABULAR_PATH)

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (21597, 29)


,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,Sale Year,House Age,house_age,is_renovated,years_since_renovation,distance_to_city_center_km,sqft_per_bedroom,sqft_per_bathroom,living_lot_ratio,price
0,7129300520,3,1.00,1180,5650,1.0,0,0,3,7,...,2014,59,60,0,60,11.972687,393.333333,1180.000000,0.208850,221900.0
1,6414100192,3,2.25,2570,7242,2.0,0,0,3,7,...,2014,63,64,1,24,12.802819,856.666667,1142.222222,0.354874,538000.0
2,5631500400,2,1.00,770,10000,1.0,0,0,3,6,...,2015,82,82,0,82,16.416960,385.000000,770.000000,0.077000,180000.0
3,2487200875,4,3.00,1960,5000,1.0,0,0,5,7,...,2014,49,50,0,50,10.538233,490.000000,653.333333,0.392000,604000.0
4,1954400510,3,2.00,1680,8080,1.0,0,0,3,8,...,2015,28,28,0,28,21.553979,560.000000,840.000000,0.207921,510000.0


In [4]:
print("Available columns:")
print(df.columns.tolist())

Available columns:
['id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'Sale Year', 'House Age', 'house_age', 'is_renovated', 'years_since_renovation', 'distance_to_city_center_km', 'sqft_per_bedroom', 'sqft_per_bathroom', 'living_lot_ratio', 'price']


In [5]:
TARGET = "price"

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' was not found. "
        f"Available columns: {df.columns.tolist()}"
    )

print("Target variable:", TARGET)
print("Target dtype:", df[TARGET].dtype)

Target variable: price
Target dtype: float64


In [6]:
print("Target statistics:")
display(df[TARGET].describe())

Target statistics:


count    2.159700e+04
mean     5.116886e+05
std      2.499386e+05
min      7.800000e+04
25%      3.220000e+05
50%      4.500000e+05
75%      6.450000e+05
max      1.129575e+06
Name: price, dtype: float64

In [7]:
candidate_features = [
    "bedrooms",
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "floors",
    "waterfront",
    "view",
    "condition",
    "grade",
    "sqft_above",
    "sqft_basement",
    "yr_built",
    "yr_renovated",
    "house_age",
    "distance_to_city_center_km"
]

available_features = [feature for feature in candidate_features if feature in df.columns]

missing_features = [feature for feature in candidate_features if feature not in df.columns]

print("Available baseline features:")
print(available_features)

print("\nFeatures not present in dataset:")
print(missing_features)

Available baseline features:
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'house_age', 'distance_to_city_center_km']

Features not present in dataset:
[]


In [8]:
X = df[available_features].copy()
y = df[TARGET].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (21597, 15)
Target shape: (21597,)


In [10]:
print("Missing values in features:")

display(X.isnull().sum())

print("Total missing feature values:", X.isnull().sum().sum())

print("Missing target values:", y.isnull().sum())

Missing values in features:


bedrooms                      0
bathrooms                     0
sqft_living                   0
sqft_lot                      0
floors                        0
waterfront                    0
view                          0
condition                     0
grade                         0
sqft_above                    0
sqft_basement                 0
yr_built                      0
yr_renovated                  0
house_age                     0
distance_to_city_center_km    0
dtype: int64

Total missing feature values: 0
Missing target values: 0


In [11]:
valid_mask = X.notnull().all(axis=1) & y.notnull()

X = X.loc[valid_mask].copy()
y = y.loc[valid_mask].copy()

print("Final feature shape:", X.shape)
print("Final target shape:", y.shape)

Final feature shape: (21597, 15)
Final target shape: (21597,)


## Train/Test Split

The cleaned dataset is divided into training and holdout subsets using an 80/20 split.

The training dataset is used to fit the Linear Regression model, while the holdout dataset remains unseen during training and will be used for model evaluation.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training samples:", len(X_train))
print("Holdout samples:", len(X_test))

print("\nTraining percentage:", round(len(X_train) / len(X) * 100, 2), "%")

print("Holdout percentage:", round(len(X_test) / len(X) * 100, 2), "%")

Training samples: 17277
Holdout samples: 4320

Training percentage: 80.0 %
Holdout percentage: 20.0 %


In [13]:
print("Training target statistics:")
display(y_train.describe())

print("\nHoldout target statistics:")
display(y_test.describe())

Training target statistics:


count    1.727700e+04
mean     5.123623e+05
std      2.501955e+05
min      7.800000e+04
25%      3.240000e+05
50%      4.500000e+05
75%      6.455000e+05
max      1.129575e+06
Name: price, dtype: float64


Holdout target statistics:


count    4.320000e+03
mean     5.089944e+05
std      2.489190e+05
min      8.300000e+04
25%      3.198000e+05
50%      4.500000e+05
75%      6.375000e+05
max      1.129575e+06
Name: price, dtype: float64

## Linear Regression Baseline

A Linear Regression model is used as the first traditional machine learning benchmark.

Feature standardization is applied before regression so that numerical features with different scales can be handled consistently.

In [14]:
linear_regression_pipeline = Pipeline(steps=[("scaler", StandardScaler()),("model", LinearRegression())])

print(linear_regression_pipeline)

Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())])


In [15]:
linear_regression_pipeline.fit(X_train,y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [16]:
y_pred = linear_regression_pipeline.predict(X_test)

print("Number of predictions:", len(y_pred))

print("\nFirst 10 predictions:")
print(y_pred[:10])

Number of predictions: 4320

First 10 predictions:
[264782.60462346 373801.54059015 158438.27975735 437002.92858166
 447632.25250244 557081.65547695 194699.73052533 942969.82126097
 490116.46226029 475994.61769017]


In [17]:
prediction_results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

prediction_results["Absolute Error"] = (prediction_results["Actual Price"] - prediction_results["Predicted Price"]).abs()

display(prediction_results.head(10))

,Actual Price,Predicted Price,Absolute Error
0,132500.0,264782.604623,132282.604623
1,415000.0,373801.540590,41198.459410
2,494000.0,158438.279757,335561.720243
3,355000.0,437002.928582,82002.928582
4,606000.0,447632.252502,158367.747498
5,640000.0,557081.655477,82918.344523
6,256500.0,194699.730525,61800.269475
7,959000.0,942969.821261,16030.178739
8,445000.0,490116.462260,45116.462260
9,254000.0,475994.617690,221994.617690


In [18]:
MODEL_PATH = os.path.join(MODEL_DIR, "linear_regression_baseline.pkl")

joblib.dump(linear_regression_pipeline, MODEL_PATH)

print("Baseline model saved successfully:")
print(MODEL_PATH)

Baseline model saved successfully:
e:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\models\linear_regression_baseline.pkl


In [19]:
WEEK2_DATA_DIR = os.path.join(DATA_DIR,"week-2")

os.makedirs(WEEK2_DATA_DIR,exist_ok=True)

In [20]:
X_train.to_csv(os.path.join(WEEK2_DATA_DIR, "X_train.csv"),index=False)

X_test.to_csv(os.path.join(WEEK2_DATA_DIR, "X_test.csv"),index=False)

y_train.to_csv(os.path.join(WEEK2_DATA_DIR, "y_train.csv"),index=False)

y_test.to_csv(os.path.join(WEEK2_DATA_DIR, "y_test.csv"),index=False)

print("Training and holdout datasets saved successfully.")

Training and holdout datasets saved successfully.


In [21]:
feature_config = pd.DataFrame({
    "Feature": available_features,
    "Purpose": ["Property characteristics" if feature not in ["house_age","distance_to_city_center_km"] 
                else "Temporal property feature" 
                if feature == "house_age" else "Location feature" for feature in available_features
    ]
})

display(feature_config)

,Feature,Purpose
0,bedrooms,Property characteristics
1,bathrooms,Property characteristics
2,sqft_living,Property characteristics
3,sqft_lot,Property characteristics
4,floors,Property characteristics
5,waterfront,Property characteristics
6,view,Property characteristics
7,condition,Property characteristics
8,grade,Property characteristics
9,sqft_above,Property characteristics


In [22]:
feature_config.to_csv(os.path.join(WEEK2_DATA_DIR,"linear_regression_features.csv"),index=False)

print("Feature configuration saved.")

Feature configuration saved.


In [23]:
print("=" * 65)
print("WEEK 2 – DAY 2 VALIDATION")
print("=" * 65)

print(f"Total records       : {len(X):,}")
print(f"Training records    : {len(X_train):,}")
print(f"Holdout records     : {len(X_test):,}")
print(f"Features used       : {len(available_features)}")
print(f"Target variable     : {TARGET}")

print(f"\nTraining split      : " f"{len(X_train) / len(X) * 100:.1f}%")

print(f"Holdout split       : " f"{len(X_test) / len(X) * 100:.1f}%")

print("\nMissing values in X_train:", X_train.isnull().sum().sum())

print("Missing values in X_test :", X_test.isnull().sum().sum())

print("\nLinear Regression     : TRAINED")
print("Model file            :", MODEL_PATH)

print("\nDay 2 completed successfully.")

WEEK 2 – DAY 2 VALIDATION
Total records       : 21,597
Training records    : 17,277
Holdout records     : 4,320
Features used       : 15
Target variable     : price

Training split      : 80.0%
Holdout split       : 20.0%

Missing values in X_train: 0
Missing values in X_test : 0

Linear Regression     : TRAINED
Model file            : e:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\models\linear_regression_baseline.pkl

Day 2 completed successfully.
